# Compare with the best small open models on physics

Runs four well-known open models of about 2B parameters on **the same 1,142 held-out UGPhysics questions** and the same 210 MATH-500 problems that our model was tested on, with the same prompts and grader. The final report puts all of them side by side with paired significance tests.

| Job | Model | Why it is here | Mode and decoding |
|---|---|---|---|
| `qwen3-1.7b` | Qwen/Qwen3-1.7B | the most used open model of this size | thinking, T 0.6, top-p 0.95, top-k 20 (Qwen's recommended thinking settings), 8,192 tokens |
| `megascience-1.7b` | MegaScience/Qwen3-1.7B-MegaScience | fine-tuned on the same science data family as ours: the most direct rival | greedy, 4,096 tokens |
| `r1-distill-1.5b` | deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B | the best-known small reasoning model | T 0.6, top-p 0.95, instructions in the user turn (DeepSeek's advice), 8,192 tokens |
| `nemotron-1.5b` | nvidia/OpenReasoning-Nemotron-1.5B | a newer reasoning distill reported strong on science (GPQA) | T 0.6, top-p 0.95, instructions in the user turn, 8,192 tokens |

**Fairness rules.** Every model gets the official UGPhysics prompt and judge and its developer's recommended decoding. Reasoning models get **8,192** new tokens, the UGPhysics paper's budget for o1-style models, twice our model's 4,096. Greedy decoding makes reasoning distills loop, so they are sampled as their developers recommend (one sample per question, fixed seed). Our model and base Qwen3.5-2B keep their existing results (greedy, 4,096 tokens). If ours still wins, the claim is strong; if not, the report says so.

**Speed.** These runs use vLLM, a fast engine, instead of plain transformers: roughly 1 to 3 T4-hours per model for both benchmarks. Qwen3.5-2B itself cannot use it here, which is why our model's earlier results are reused.

**How to run.** One job per session: set `JOB`, run sections 1 to 4. A job is safe to interrupt; run the same cell again and it continues. Never run the same job in two sessions at once. When all four are done, section 5 (no GPU) builds the comparison. Needs `physeval.py` **version 1.5** in the Drive folder.

## 1. Settings

In [ ]:
WORKDIR = "/content/drive/MyDrive/physics_eval"
JOB = "qwen3-1.7b"      # "qwen3-1.7b", "megascience-1.7b", "r1-distill-1.5b" or "nemotron-1.5b"

SAMPLED = "--temperature 0.6 --top-p 0.95"
MODELS = {
    "qwen3-1.7b":       dict(run="cmp-qwen3-1.7b-think", model="Qwen/Qwen3-1.7B", budget=8192,
                             flags=f"--enable-thinking true {SAMPLED} --top-k 20"),
    "megascience-1.7b": dict(run="cmp-qwen3-1.7b-megascience", model="MegaScience/Qwen3-1.7B-MegaScience", budget=4096,
                             flags=""),
    "r1-distill-1.5b":  dict(run="cmp-r1-distill-qwen-1.5b", model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", budget=8192,
                             flags=f"{SAMPLED} --system-in-user"),
    "nemotron-1.5b":    dict(run="cmp-openreasoning-nemotron-1.5b", model="nvidia/OpenReasoning-Nemotron-1.5B", budget=8192,
                             flags=f"{SAMPLED} --system-in-user"),
}
OURS_TEST, OURS_DEV = "ft-qwen-v1.2-lora-s647-x50-test", "ft-qwen-v1.2-lora-s647-x50"   # existing runs
BASE_TEST, BASE_DEV = "qwen35-2b-nothink-test", "qwen35-2b-nothink"
SLICES = "ugphysics=100:200,math500=30"      # exactly the questions our model was tested on
assert JOB in MODELS, f"JOB must be one of {list(MODELS)}"
M = MODELS[JOB]
GEN = (f'--workdir "{WORKDIR}" --run {M["run"]} --model {M["model"]} {M["flags"]} --backend vllm '
       f'--max-new-tokens {M["budget"]} --batch-size 256')
print(JOB, "->", M["run"])

## 2. Mount Drive and check the files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, shutil
sys.path.insert(0, WORKDIR)
import physeval
assert physeval.VERSION >= "1.5", "replace physeval.py with version 1.5"
for r in (OURS_TEST, BASE_TEST, OURS_DEV, BASE_DEV):
    assert os.path.exists(f"{WORKDIR}/runs/{r}/graded_" + ("ugphysics" if r.endswith("-test") else "math500") + ".jsonl"), f"missing results of {r}"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install vLLM (about 5 minutes)
If Colab offers to restart the session after this cell, accept, then run sections 1 and 2 again and continue with section 4.

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U vllm
!pip -q install "antlr4-python3-runtime==4.11.1" chardet
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"

## 4. Run the job
**4a. Smoke test (2 to 5 minutes).** Answers 8 questions and prints the exact prompt and the first answer. Check that the prompt is a normal chat prompt and that the answer ends with a `\boxed{...}` result (after `</think>` for reasoning models). Nothing is saved as a result.

In [ ]:
!python "{WORKDIR}/physeval.py" generate {GEN} --benchmarks ugphysics --per-group "ugphysics=100:200" --smoke

**4b. Full run (about 1 to 3 hours).** Both benchmarks, then grading. Safe to stop; run it again to continue.

In [ ]:
!python "{WORKDIR}/physeval.py" generate {GEN} --benchmarks ugphysics,math500 --per-group "{SLICES}"
!python "{WORKDIR}/physeval.py" grade --workdir "{WORKDIR}" --run {M["run"]}
print("JOB", JOB, "finished")

## 5. Comparison report (no GPU; after all four jobs)
Builds two reports (UGPhysics held-out, MATH-500) and a short leaderboard with paired tests against our model. Jobs not finished yet are simply left out. Send me `results/compare_leaderboard.md` and the two `summary_compare_*.md` files.

In [ ]:
import subprocess, os, json
done = [m["run"] for m in MODELS.values() if os.path.exists(f"{WORKDIR}/runs/{m['run']}/graded_ugphysics.jsonl")]
print("finished competitor runs:", done)
reports = {}
for bench, runs in (("ugphysics", [OURS_TEST, BASE_TEST] + done), ("math500", [OURS_DEV, BASE_DEV] + done)):
    r = subprocess.run([sys.executable, f"{WORKDIR}/physeval.py", "report", "--workdir", WORKDIR,
                        "--run", ",".join(runs), "--benchmarks", bench], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-800:]
    shutil.copy(f"{WORKDIR}/results/summary.md", f"{WORKDIR}/results/summary_compare_{bench}.md")
    shutil.copy(f"{WORKDIR}/results/summary.json", f"{WORKDIR}/results/summary_compare_{bench}.json")
    reports[bench] = json.load(open(f"{WORKDIR}/results/summary.json"))

NAMES = {OURS_TEST: "**This model** (Qwen3.5-2B + half-strength LoRA)", OURS_DEV: "**This model** (Qwen3.5-2B + half-strength LoRA)",
         BASE_TEST: "Qwen3.5-2B, non-thinking (base)", BASE_DEV: "Qwen3.5-2B, non-thinking (base)"}
for k, m in MODELS.items():
    NAMES[m["run"]] = m["model"]
def acc(js, run, bench):
    b = js["runs"].get(run, {}).get("benchmarks", {}).get(bench)
    if not b:
        return None, ""
    r = b["rule"]
    return 100 * r["weighted"], "%.1f [%.1f, %.1f]" % (100 * r["weighted"], 100 * r["ci"][0], 100 * r["ci"][1])
def vs_ours(js, ours, run, bench, metric):
    for h in js["head_to_head"]:
        if h["bench"] == bench and h["metric"] == metric and {h["A"], h["B"]} == {ours, run}:
            sign = 1 if h["A"] == ours else -1
            d, lo, hi = sign * h["diff"] * 100, sign * h["ci"][0] * 100, sign * h["ci"][1] * 100
            lo, hi = min(lo, hi), max(lo, hi)
            wins, losses = (h["a_only"], h["b_only"]) if sign == 1 else (h["b_only"], h["a_only"])
            return "%+.1f [%+.1f, %+.1f]" % (d, lo, hi), "%d / %d" % (wins, losses), "%.4f" % h["p"]
    return "", "", ""
ug, mt = reports["ugphysics"], reports["math500"]
rows = []
for test_run, dev_run in [(OURS_TEST, OURS_DEV), (BASE_TEST, BASE_DEV)] + [(r, r) for r in done]:
    a_ug, t_ug = acc(ug, test_run, "ugphysics")
    a_mt, t_mt = acc(mt, dev_run, "math500")
    d, wl, p = ("-", "-", "-") if test_run == OURS_TEST else vs_ours(ug, OURS_TEST, test_run, "ugphysics", "rule accuracy")
    cfg = ug["runs"][test_run]["meta"]["cfg"]
    mode = ("greedy" if not cfg.get("temperature") else "T %s" % cfg["temperature"]) + ", %d tok" % cfg["max_new_tokens"]
    rows.append((a_ug or -1, "| %s | %s | %s | %s | %s | %s | %s |" % (NAMES[test_run], mode, t_ug, d, wl, p, t_mt or "-")))
rows.sort(key=lambda r: -r[0])
out = ["# Physics comparison: about 2B open models", "",
       "UGPhysics (English) held-out test: 1,142 questions (ranks 100-199 in 12 subjects), official prompt and rule judge,"
       " accuracy weighted by subject. MATH-500: 210 problems. 95% intervals in brackets. 'Ours minus model' and"
       " wins/losses are paired on the same questions (exact McNemar p).", "",
       "| Model | Decoding | UGPhysics held-out % | Ours minus model | Ours wins / losses | p | MATH-500 % |",
       "|---|---|---|---|---|---|---|"] + [r[1] for r in rows]
open(f"{WORKDIR}/results/compare_leaderboard.md", "w").write("\n".join(out) + "\n")
from IPython.display import Markdown, display
display(Markdown("\n".join(out)))
print("saved: results/compare_leaderboard.md, results/summary_compare_ugphysics.md, results/summary_compare_math500.md")